In [1]:
import time
import pandas as pd
from tqdm import tqdm
from mistralai.client import Mistral

In [2]:
BASE_PATH = "../"

In [5]:
import fitz 

def parse_pdf_to_slides(pdf_path):
    slides = []
    doc = fitz.open(pdf_path)
    
    for i, page in enumerate(doc):
        text = page.get_text("text")
        if text and text.strip():
            import re
            clean_text = re.sub(r' {2,}', ' ', text.strip())
            
            slides.append({"id": f"Slide {i+1}", "content": clean_text})
            
    doc.close()
    return slides

In [6]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

def parse_code_to_blocks(file_path):
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    
    ext = file_path.split('.')[-1].lower()
    
    lang_map = {
        'py': Language.PYTHON,
        'cpp': Language.CPP,
        'c': Language.C,
        'h': Language.CPP,
        'hpp': Language.CPP,
        'cu': Language.CPP,
        'cuh': Language.CPP,
        'java': Language.JAVA,
        'go': Language.GO,
        'rs': Language.RUST,
    }
    
    lang = lang_map.get(ext)
    
    if lang:
        splitter = RecursiveCharacterTextSplitter.from_language(
            language=lang, 
            chunk_size=1500, 
            chunk_overlap=150
        )
        blocks = splitter.split_text(content)
    else:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1500,
            chunk_overlap=150
        )
        blocks = splitter.split_text(content)
        
    formatted_blocks = []
    for i, b in enumerate(blocks):
        if len(b.strip()) > 20:
            formatted_blocks.append({
                "id": f"{os.path.basename(file_path)}_block{i+1}",
                "content": b.strip()
            })
            
    return formatted_blocks

In [7]:
def parse_all_raw_data(base_path, subdirs):
    all_slides = {}
    for subdir in subdirs:
        folder_path = os.path.join(base_path, "data", "raw", subdir)
        if not os.path.exists(folder_path): 
            continue
        
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            if not os.path.isfile(file_path): 
                continue
            if filename.endswith(".mp3") or filename == "labels.json":
                continue
                
            elif filename.endswith(".pdf"):
                slides = parse_pdf_to_slides(file_path)
                for slide in slides:
                    all_slides[f"{subdir}_{slide['id']}"] = slide['content']
                    
            else: 
                blocks = parse_code_to_blocks(file_path)
                for block in blocks:
                    all_slides[f"{subdir}_{block['id']}"] = block['content']
                    
    return all_slides